# CycleGAN 消融实验分析工具

## 📊 消融实验结果对比与分析

这个 notebook 提供了：
- 📈 训练曲线可视化
- 📊 性能指标对比
- 🖼️ 生成图像对比
- 📝 自动生成报告

---

## 1️⃣ 环境设置

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
from datetime import datetime

# 设置项目路径
project_root = Path('/Users/leon/Desktop/cycleGAN')
checkpoints_dir = project_root / 'checkpoints'
results_dir = project_root / 'results_ablation'

os.chdir(project_root)

print(f"✅ 项目路径: {project_root}")
print(f"✅ 检查点目录: {checkpoints_dir}")
print(f"✅ 结果目录: {results_dir}")

## 2️⃣ 加载消融配置

In [ ]:
# 加载消融实验配置
config_path = project_root / 'ablation_configs.json'

if config_path.exists():
    with open(config_path, 'r') as f:
        ablation_config = json.load(f)
    print("✅ 加载配置成功\n")
else:
    print("❌ 配置文件不存在")
    print("请先运行: python ablation_study.py --gen-config")
    ablation_config = {}

# 显示基准配置
print("📋 基准配置 (Full CycleGAN):")
for key, value in ablation_config.get('base_experiment', {}).items():
    if key != 'name' and key != 'description':
        print(f"  {key:20s}: {value}")

# 显示所有实验
print(f"\n📊 实验列表 ({len(ablation_config.get('experiments', []))} 个):")
for i, exp in enumerate(ablation_config.get('experiments', []), 1):
    print(f"  {i}. {exp['name']}")

## 3️⃣ 收集实验结果

In [ ]:
def load_loss_history(exp_name, checkpoints_dir):
    """从检查点目录加载损失历史"""
    exp_dir = checkpoints_dir / exp_name.replace(' ', '_').lower()
    loss_file = exp_dir / 'loss_log.txt'
    
    losses = {'epoch': [], 'loss_G': [], 'loss_D': [], 'loss_cycle': []}
    
    if loss_file.exists():
        try:
            with open(loss_file, 'r') as f:
                for line in f:
                    data = json.loads(line)
                    losses['epoch'].append(data.get('epoch', 0))
                    losses['loss_G'].append(data.get('loss_G', 0))
                    losses['loss_D'].append(data.get('loss_D', 0))
                    losses['loss_cycle'].append(data.get('loss_cycle', 0))
        except:
            pass
    
    return losses

# 收集所有实验的结果
experiments_summary = {}

# 基准实验
baseline_name = ablation_config.get('base_experiment', {}).get('name', 'Full CycleGAN')
baseline_loss = load_loss_history(baseline_name, checkpoints_dir)
experiments_summary[baseline_name] = baseline_loss

# 消融实验
for exp in ablation_config.get('experiments', []):
    exp_name = exp['name']
    exp_loss = load_loss_history(exp_name, checkpoints_dir)
    experiments_summary[exp_name] = exp_loss

print(f"✅ 加载了 {len(experiments_summary)} 个实验的结果")
for exp_name in experiments_summary.keys():
    print(f"  - {exp_name}")

## 4️⃣ 可视化训练曲线

In [ ]:
# 创建训练曲线对比图
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('CycleGAN 消融实验 - 训练曲线对比', fontsize=16, fontweight='bold')

colors = plt.cm.Set3(np.linspace(0, 1, len(experiments_summary)))

# 生成器损失
ax = axes[0, 0]
for (exp_name, losses), color in zip(experiments_summary.items(), colors):
    if losses['loss_G']:
        ax.plot(losses['epoch'], losses['loss_G'], label=exp_name, linewidth=2, color=color, alpha=0.8)
ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('Loss', fontsize=11)
ax.set_title('生成器损失 (Generator Loss)', fontweight='bold')
ax.legend(loc='best', fontsize=9)
ax.grid(True, alpha=0.3)

# 判别器损失
ax = axes[0, 1]
for (exp_name, losses), color in zip(experiments_summary.items(), colors):
    if losses['loss_D']:
        ax.plot(losses['epoch'], losses['loss_D'], label=exp_name, linewidth=2, color=color, alpha=0.8)
ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('Loss', fontsize=11)
ax.set_title('判别器损失 (Discriminator Loss)', fontweight='bold')
ax.legend(loc='best', fontsize=9)
ax.grid(True, alpha=0.3)

# 循环一致性损失
ax = axes[1, 0]
for (exp_name, losses), color in zip(experiments_summary.items(), colors):
    if losses['loss_cycle']:
        ax.plot(losses['epoch'], losses['loss_cycle'], label=exp_name, linewidth=2, color=color, alpha=0.8)
ax.set_xlabel('Epoch', fontsize=11)
ax.set_ylabel('Loss', fontsize=11)
ax.set_title('循环一致性损失 (Cycle Loss)', fontweight='bold')
ax.legend(loc='best', fontsize=9)
ax.grid(True, alpha=0.3)

# 总损失对比
ax = axes[1, 1]
ax.axis('off')

# 显示配置汇总
summary_text = "实验配置汇总\n" + "="*30 + "\n\n"
for exp_name in experiments_summary.keys():
    summary_text += f"✓ {exp_name}\n"

ax.text(0.1, 0.5, summary_text, fontsize=10, family='monospace',
        verticalalignment='center', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig(str(results_dir / 'training_curves_comparison.png'), dpi=150, bbox_inches='tight')
print("✅ 训练曲线已保存到:", results_dir / 'training_curves_comparison.png')
plt.show()

## 5️⃣ 参数对比表

In [ ]:
# 创建参数对比表
comparison_data = []

# 基准
base_exp = ablation_config.get('base_experiment', {})
comparison_data.append({
    '实验': base_exp.get('name', 'Full CycleGAN'),
    '描述': base_exp.get('description', ''),
    'lambda_A': base_exp.get('lambda_A', 10.0),
    'lambda_B': base_exp.get('lambda_B', 10.0),
    'lambda_identity': base_exp.get('lambda_identity', 0.5)
})

# 消融实验
for exp in ablation_config.get('experiments', []):
    comparison_data.append({
        '实验': exp.get('name', ''),
        '描述': exp.get('description', ''),
        'lambda_A': exp.get('lambda_A', 0),
        'lambda_B': exp.get('lambda_B', 0),
        'lambda_identity': exp.get('lambda_identity', 0)
    })

df_comparison = pd.DataFrame(comparison_data)

# 显示表格
print("\n📊 消融实验参数对比\n")
print(df_comparison.to_string(index=False))

# 保存为 CSV
csv_path = results_dir / 'ablation_comparison.csv'
df_comparison.to_csv(csv_path, index=False, encoding='utf-8')
print(f"\n✅ 对比表已保存到: {csv_path}")

## 6️⃣ 超参数影响分析

In [ ]:
# 可视化超参数影响
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('CycleGAN 消融实验 - 超参数影响分析', fontsize=14, fontweight='bold')

exp_names = df_comparison['实验'].tolist()
x_pos = np.arange(len(exp_names))
width = 0.25

# lambda_A
ax = axes[0]
lambda_a_values = df_comparison['lambda_A'].tolist()
bars = ax.bar(x_pos, lambda_a_values, width, alpha=0.8, color='steelblue')
ax.set_ylabel('λ_A', fontsize=11, fontweight='bold')
ax.set_title('Cycle Loss Weight (A→B→A)', fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels([e.replace(' ', '\n') for e in exp_names], fontsize=9)
ax.grid(axis='y', alpha=0.3)
# 添加数值标签
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.1f}', ha='center', va='bottom', fontsize=9)

# lambda_B
ax = axes[1]
lambda_b_values = df_comparison['lambda_B'].tolist()
bars = ax.bar(x_pos, lambda_b_values, width, alpha=0.8, color='darkorange')
ax.set_ylabel('λ_B', fontsize=11, fontweight='bold')
ax.set_title('Cycle Loss Weight (B→A→B)', fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels([e.replace(' ', '\n') for e in exp_names], fontsize=9)
ax.grid(axis='y', alpha=0.3)
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.1f}', ha='center', va='bottom', fontsize=9)

# lambda_identity
ax = axes[2]
lambda_idt_values = df_comparison['lambda_identity'].tolist()
bars = ax.bar(x_pos, lambda_idt_values, width, alpha=0.8, color='seagreen')
ax.set_ylabel('λ_identity', fontsize=11, fontweight='bold')
ax.set_title('Identity Loss Weight', fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels([e.replace(' ', '\n') for e in exp_names], fontsize=9)
ax.grid(axis='y', alpha=0.3)
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.1f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(str(results_dir / 'hyperparameter_comparison.png'), dpi=150, bbox_inches='tight')
print("✅ 超参数对比图已保存")
plt.show()

## 7️⃣ 生成对比分析报告

In [ ]:
# 生成详细的分析报告
report_content = f"""
# CycleGAN 消融实验分析报告

**生成时间**: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}

---

## 📋 实验概述

本消融实验旨在理解 CycleGAN 中不同损失函数组件对模型性能的贡献。

### 测试的损失函数

1. **GAN 损失** (Generator Adversarial Loss)
   - 使生成图像更加真实

2. **循环一致性损失** (Cycle Consistency Loss)  
   - 确保 A→B→A 和 B→A→B 映射的一致性
   - 权重: λ_A, λ_B

3. **恒等映射损失** (Identity Loss)
   - 在源目标颜色接近时保留图像
   - 权重: λ_identity

---

## 🧪 实验配置

### 基准模型 (Full CycleGAN)

| 参数 | 值 |
|------|----|
| λ_A | {base_exp.get('lambda_A', 10.0)} |
| λ_B | {base_exp.get('lambda_B', 10.0)} |
| λ_identity | {base_exp.get('lambda_identity', 0.5)} |

### 消融实验
"""

for i, exp in enumerate(ablation_config.get('experiments', []), 1):
    report_content += f"""
#### 实验 {i}: {exp['name']}
- **描述**: {exp.get('description', 'N/A')}
- **λ_A**: {exp.get('lambda_A', 0)}
- **λ_B**: {exp.get('lambda_B', 0)}
- **λ_identity**: {exp.get('lambda_identity', 0)}
"""

report_content += f"""

---

## 📊 关键发现

### 循环一致性损失的重要性

循环一致性损失是 CycleGAN 的核心组件，确保：
- 内容结构的保留
- 图像转换的可逆性
- 收敛的稳定性

**预期**：移除此损失会导致严重的图像失真。

### 恒等映射损失的影响

恒等映射损失有助于：
- 在源目标域相似时保留颜色
- 减少不必要的转换
- 改善相近域之间的转换质量

**预期**：移除此损失会导致颜色转换过度。

### GAN 损失的作用

GAN 损失提供：
- 生成图像的真实感
- 纹理和细节的丰富性
- 视觉质量的提升

**预期**：移除此损失会导致生成图像平滑但缺乏细节。

---

## 📈 建议

1. **不移除循环一致性损失**
   - 在任何情况下都应保留
   - 可以调整权重 (λ_A, λ_B)

2. **根据任务调整恒等映射损失**
   - 跨域颜色差异大：减小 λ_identity
   - 跨域颜色相似：增大 λ_identity

3. **平衡 GAN 损失和循环损失**
   - 过高的循环损失：图像保守，转换不充分
   - 过低的循环损失：图像失真，无法保留内容

---

## 📝 实验设置

**数据集**: handbags (或自定义数据集)
**训练轮数**: 200 (基准) + 200 (衰减)
**Batch 大小**: 1
**学习率**: 0.0002
**优化器**: Adam (β1=0.5, β2=0.999)

---

## 📊 可视化

- 📈 **训练曲线**: training_curves_comparison.png
- 📊 **超参数对比**: hyperparameter_comparison.png
- 📋 **参数汇总**: ablation_comparison.csv

---

## 🔍 后续工作

1. **定量评估**
   - 计算 FID、IS、LPIPS 等指标
   - 对比不同配置的量化性能

2. **定性分析**
   - 可视化对比生成图像
   - 分析各配置的优缺点

3. **参数优化**
   - 基于消融实验结果调整超参数
   - 针对特定任务进行微调

---

**报告生成者**: CycleGAN Ablation Study Framework  
**联系**: 查看 ABLATION_STUDY.md 获取更多信息
"""

# 保存报告
report_path = results_dir / f"ablation_analysis_report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.md"
os.makedirs(report_path.parent, exist_ok=True)

with open(report_path, 'w', encoding='utf-8') as f:
    f.write(report_content)

print(f"✅ 分析报告已生成: {report_path}")
print(f"\n报告预览:\n{'='*60}")
print(report_content[:800] + "...")

## 8️⃣ 后续步骤

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════╗
║         🎉 消融实验分析完成！                                    ║
╚══════════════════════════════════════════════════════════════════╝

📊 已生成的文件：
  ✅ training_curves_comparison.png - 训练曲线对比
  ✅ hyperparameter_comparison.png - 超参数对比图  
  ✅ ablation_comparison.csv - 参数对比表
  ✅ ablation_analysis_report_*.md - 分析报告

🚀 后续步骤：

1️⃣ 定量评估
   计算 FID、IS、LPIPS 等评估指标
   python scripts/compute_fid.py --real_path ./datasets/handbags/testB --fake_path results/checkpoints

2️⃣ 对比生成图像
   可视化不同实验的输出图像
   查看 results 文件夹中的对比图

3️⃣ 分析训练动态
   研究损失曲线的收敛特性
   识别最优训练轮数

4️⃣ 参数优化
   基于消融结果调整超参数
   重新训练最优模型

5️⃣ 撰写论文
   使用消融实验结果支持论文论证
   在相关部分引用实验结果

📚 参考资源：
  • ABLATION_STUDY.md - 详细指南
  • ablation_study.py - 实验框架
  • models/cycle_gan_ablation_model.py - 增强模型

💡 提示：
  • 每个实验运行多次取平均值可提高结果可靠性
  • 保存所有实验的日志便于后续参考
  • 记录每个实验的关键观察结论

🎯 好运！祝消融实验顺利！
""")